<a href="https://colab.research.google.com/github/KarimoWusal/DoSi/blob/main/Code_version_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install rdkit transformers accelerate bitsandbytes shap

In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/Togrulla girdiyimiz proyekt/tox21.csv'
df = pd.read_csv(file_path)

print(f"Total molecules in raw dataset: {len(df)}")

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')

def smiles_to_fingerprint(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return None
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
        return np.array(fp)
    except:
        return None

In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

tox21_targets = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
    'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53'
]

endpoint_models = {}

print("Training Multi-Endpoint Array...")
for target in tox21_targets:
    if target not in df.columns: continue

    df_target = df[['smiles', target]].dropna()
    df_target['features'] = df_target['smiles'].apply(smiles_to_fingerprint)
    df_target = df_target.dropna(subset=['features'])

    X_target = np.stack(df_target['features'].values)
    y_target = df_target[target].values

    X_train, X_test, y_train, y_test = train_test_split(X_target, y_target, test_size=0.2, random_state=42)

    model = XGBClassifier(
        eval_metric='logloss', scale_pos_weight=15, learning_rate=0.05,
        max_depth=4, subsample=0.8, colsample_bytree=0.8, n_estimators=300
    )
    model.fit(X_train, y_train)
    endpoint_models[target] = model

print("All 12 models trained and stored in memory!")

In [ ]:
def extract_chemical_fragments(smiles, important_bits, radius=2, nBits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return {}

    bit_info = {}
    AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits, bitInfo=bit_info)

    fragments = {}
    for bit in important_bits:
        if bit in bit_info:
            atom_idx, bit_radius = bit_info[bit][0]
            if bit_radius == 0:
                atom_indices = [atom_idx]
            else:
                bond_indices = Chem.FindAtomEnvironmentOfRadiusN(mol, bit_radius, atom_idx)
                atom_indices = set()
                for bond_idx in bond_indices:
                    bond = mol.GetBondWithIdx(bond_idx)
                    atom_indices.add(bond.GetBeginAtomIdx())
                    atom_indices.add(bond.GetEndAtomIdx())
                atom_indices = list(atom_indices)
            fragments[bit] = Chem.MolFragmentToSmiles(mol, atom_indices)
    return fragments

In [ ]:
import torch
from transformers import pipeline
import numpy as np
import shap

print("Loading fast 1.5B LLM... (Executing zero-trust architecture)")

# 1. Load the lightweight 1.5B model
pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16
)

# 2. Pick a sample molecule from the dataset
# We drop NaNs just to make sure we grab a valid SMILES string
sample_smiles = df['smiles'].dropna().iloc[12]
sample_features = np.array(smiles_to_fingerprint(sample_smiles)).reshape(1, -1)

print(f"\n--- Pipeline-Wide Failure Filter for SMILES: {sample_smiles} ---")

# 3. Run the molecule through ALL 12 models and isolate the failures
high_risk_endpoints = {}
low_risk_endpoints = []

print("Running 12-endpoint diagnostic array...")
for target, model in endpoint_models.items():
    fail_prob = model.predict_proba(sample_features)[0][1] * 100

    if fail_prob > 50.0:
        # Only run SHAP and extract fragments if it actually failed the screen!
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(sample_features)[0]
        top_indices = np.argsort(shap_values)[-3:][::-1].tolist()
        fragments = extract_chemical_fragments(sample_smiles, top_indices)

        high_risk_endpoints[target] = {
            'probability': fail_prob,
            'fragments': fragments
        }
    else:
        low_risk_endpoints.append(target)

# 4. Build the dynamic, context-aware prompt
system_prompt = "You are a rigid, factual AI Data Translator for a biotech firm. You do not use conversational filler. You strictly output the requested Markdown template."

if len(high_risk_endpoints) == 0:
    # Scenario A: The molecule passed everything
    user_prompt = f"""
    Translate the following deterministic ML output into a professional risk assessment.

    DATA: Molecule passed all {len(low_risk_endpoints)} evaluated Tox21 endpoints with <50% failure probability.

    INSTRUCTIONS: Output EXACTLY in the following format. Do not add introductions.

    ### 1. Multi-Endpoint Risk Assessment
    [State that the molecule was screened across {len(low_risk_endpoints)} clinical liability endpoints and passed all of them, indicating a low overall toxicity risk profile.]

    ### 2. Structural Attribution
    [State that no severe structural liabilities were flagged by the XGBoost pipeline across any biological targets.]

    ### 3. Mitigation Directive
    [State that no immediate structural mitigation is required and the compound is mathematically cleared for further development.]
    """
else:
    # Scenario B: The molecule failed one or more endpoints
    failed_data_str = ""
    for target, data in high_risk_endpoints.items():
        failed_data_str += f"\n- {target}: {data['probability']:.1f}% risk. Key toxic fragments: {data['fragments']}"

    user_prompt = f"""
    Translate the following deterministic ML output into a professional risk assessment.

    DATA:
    - Molecule triggered HIGH RISK for the following endpoints: {failed_data_str}
    - Molecule passed {len(low_risk_endpoints)} other endpoints.

    INSTRUCTIONS: Output EXACTLY in the following format. Do not add introductions.

    ### 1. Multi-Endpoint Risk Assessment
    [State that the molecule failed {len(high_risk_endpoints)} out of 12 toxicity screens. List the specific failed endpoints and their exact probabilities.]

    ### 2. Structural Attribution
    [List the specific fragments flagged for each failed endpoint based on the DATA. State that SHAP analysis isolated these as the mathematical drivers of the toxicity.]

    ### 3. Mitigation Directive
    [Recommend that the medicinal chemistry team evaluate the flagged fragments for bioisosteric replacement to mitigate the specific pathway liabilities identified.]
    """

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

print("Generating strictly constrained multi-endpoint feedback...\n")
outputs = pipe(messages, max_new_tokens=400, temperature=0.01)
print(outputs[0]["generated_text"][-1]["content"])

In [ ]:
# ---------------------------------------------------------
# CELL 8: MULTI-ENDPOINT FAST INFERENCE ENGINE
# Run this as many times as you want with new molecules!
# ---------------------------------------------------------
import numpy as np
import shap

# 1. Paste your NEW molecule's SMILES string right here:
new_smiles = "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"  # Ibuprofen

print(f"\n--- Pipeline-Wide Failure Filter for SMILES: {new_smiles} ---")

# 2. Extract mathematical features
new_features = np.array(smiles_to_fingerprint(new_smiles)).reshape(1, -1)

if new_features.any() == None:
    print("Error: RDKit could not read this SMILES string. Please check the syntax.")
else:
    high_risk_endpoints = {}
    low_risk_endpoints = []

    print("Running 12-endpoint diagnostic array...")

    # 3. Screen against all 12 models
    for target, model in endpoint_models.items():
        fail_prob = model.predict_proba(new_features)[0][1] * 100

        if fail_prob > 50.0:
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(new_features)[0]
            top_indices = np.argsort(shap_values)[-3:][::-1].tolist()
            fragments = extract_chemical_fragments(new_smiles, top_indices)

            high_risk_endpoints[target] = {
                'probability': fail_prob,
                'fragments': fragments
            }
        else:
            low_risk_endpoints.append(target)

    # 4. Build the dynamic prompt
    system_prompt = "You are a rigid, factual AI Data Translator for a biotech firm. You do not use conversational filler. You strictly output the requested Markdown template."

    if len(high_risk_endpoints) == 0:
        user_prompt = f"""
        Translate the following deterministic ML output into a professional risk assessment.

        DATA: Molecule passed all {len(low_risk_endpoints)} evaluated Tox21 endpoints with <50% failure probability.

        INSTRUCTIONS: Output EXACTLY in the following format. Do not add introductions.

        ### 1. Multi-Endpoint Risk Assessment
        [State that the molecule was screened across {len(low_risk_endpoints)} clinical liability endpoints and passed all of them, indicating a low overall toxicity risk profile.]

        ### 2. Structural Attribution
        [State that no severe structural liabilities were flagged by the XGBoost pipeline across any biological targets.]

        ### 3. Mitigation Directive
        [State that no immediate structural mitigation is required and the compound is mathematically cleared for further development.]
        """
    else:
        failed_data_str = ""
        for target, data in high_risk_endpoints.items():
            failed_data_str += f"\n- {target}: {data['probability']:.1f}% risk. Key toxic fragments: {data['fragments']}"

        user_prompt = f"""
        Translate the following deterministic ML output into a professional risk assessment.

        DATA:
        - Molecule triggered HIGH RISK for the following endpoints: {failed_data_str}
        - Molecule passed {len(low_risk_endpoints)} other endpoints.

        INSTRUCTIONS: Output EXACTLY in the following format. Do not add introductions.

        ### 1. Multi-Endpoint Risk Assessment
        [State that the molecule failed {len(high_risk_endpoints)} out of 12 toxicity screens. List the specific failed endpoints and their exact probabilities.]

        ### 2. Structural Attribution
        [List the specific fragments flagged for each failed endpoint based on the DATA. State that SHAP analysis isolated these as the mathematical drivers of the toxicity.]

        ### 3. Mitigation Directive
        [Recommend that the medicinal chemistry team evaluate the flagged fragments for bioisosteric replacement to mitigate the specific pathway liabilities identified.]
        """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    print("Generating strictly constrained multi-endpoint feedback...\n")
    outputs = pipe(messages, max_new_tokens=400, temperature=0.01)
    print(outputs[0]["generated_text"][-1]["content"])